# Capítulo 11 — Inteligência artificial: uma rede neural do zero

## Missão
Dar ao rover a capacidade de **aprender**. Primeiro, uma rede neural escrita à mão com NumPy, para ver cada peça funcionando: pesos, ativação, propagação, erro e retropropagação. Depois, um classificador de verdade com scikit-learn, que reconhece dígitos escritos à mão (o livro usa TensorFlow com roupas; a ideia é a mesma, sem instalar 500 MB), e gráficos com Matplotlib.

**No fim deste caderno você terá:** o módulo `gabarito_rover/neural.py` (rede neural de 2 camadas em NumPy) e um modelo que classifica imagens com mais de 90% de acerto.

**Leitura essencial (opcional, livro em inglês):**
- AI Is a Collection of Techniques · PDF p. 378
- Understanding Neural Networks · PDF p. 388
- Building a Simple Neural Network in Python · PDF p. 392
- Creating a Machine-Learning Network for Detecting Clothes Types · PDF p. 419
- Visualizing with MatPlotLib · PDF p. 433

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_rover/
os.makedirs("../gabarito_rover", exist_ok=True)
open("../gabarito_rover/__init__.py", "a").close()        # gabarito_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_rover")

## Ideia em 30 segundos
Um neurônio faz `ativação(soma dos pesos × entradas + viés)`. Uma rede junta camadas de neurônios. Para aprender, ela:
1. faz uma previsão (*forward*);
2. mede o erro;
3. calcula quanto cada peso contribuiu para o erro (*backpropagation*);
4. ajusta um pouquinho na direção que reduz o erro (gradiente).

Repita milhares de vezes e pronto.

### 11.1 **Um neurônio.** Implemente a sigmoide `1 / (1 + e^-x)` e um neurônio com pesos `[0.8, -0.5]` e viés `0.1`. Calcule a saída para as entradas `[1, 0]`, `[0, 1]` e `[1, 1]`.

In [ ]:
import numpy as np

def sigmoide(x):
    return 1 / (1 + np.exp(-x))

w, b = np.array([0.8, -0.5]), 0.1
for x in ([1, 0], [0, 1], [1, 1]):
    print(x, round(float(sigmoide(np.dot(w, x) + b)), 3))

### 11.2 **Por que uma camada não basta.** O XOR (saída 1 quando as entradas são diferentes) não tem como ser separado por uma reta. Monte os 4 casos em `X` e `y` e desenhe-os num gráfico de dispersão com Matplotlib (cores pela classe).

In [ ]:
import matplotlib.pyplot as plt
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), s=200, cmap="coolwarm")
plt.title("XOR: nenhuma reta separa as cores"); plt.show()

### 11.3 **Rede neural do zero.** Escreva `neural.py` com a classe `RedeNeural(entradas, ocultos, saidas, semente=0)`:
- pesos `W1`, `W2` aleatórios pequenos e vieses `b1`, `b2` zerados;
- `prever(X)`: sigmoide nas duas camadas;
- `treinar(X, y, epocas=5000, taxa=1.0)`: a cada época faz o *forward*, calcula o erro quadrático médio, retropropaga (derivada da sigmoide: `s × (1 - s)`) e atualiza os pesos; guarda a perda em `self.perdas`.

_A célula grava `gabarito_rover/neural.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_rover/neural.py
"""Rede neural de 2 camadas escrita só com NumPy."""
import numpy as np


def _sig(x):
    return 1 / (1 + np.exp(-x))


class RedeNeural:
    def __init__(self, entradas: int, ocultos: int, saidas: int, semente: int = 0):
        rng = np.random.default_rng(semente)
        self.W1 = rng.normal(0, 1, (entradas, ocultos))
        self.b1 = np.zeros((1, ocultos))
        self.W2 = rng.normal(0, 1, (ocultos, saidas))
        self.b2 = np.zeros((1, saidas))
        self.perdas: list[float] = []

    def _forward(self, X):
        self.h = _sig(X @ self.W1 + self.b1)
        self.o = _sig(self.h @ self.W2 + self.b2)
        return self.o

    def prever(self, X):
        return self._forward(np.asarray(X, dtype=float))

    def treinar(self, X, y, epocas: int = 5000, taxa: float = 1.0):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        for _ in range(epocas):
            o = self._forward(X)
            erro = o - y
            self.perdas.append(float((erro ** 2).mean()))
            d_o = erro * o * (1 - o)                       # gradiente na saída
            d_h = (d_o @ self.W2.T) * self.h * (1 - self.h)  # retropropagado para a camada oculta
            self.W2 -= taxa * self.h.T @ d_o / len(X)
            self.b2 -= taxa * d_o.mean(axis=0, keepdims=True)
            self.W1 -= taxa * X.T @ d_h / len(X)
            self.b1 -= taxa * d_h.mean(axis=0, keepdims=True)
        return self


### 11.4 **Treine no XOR.** Crie `RedeNeural(2, 4, 1)`, treine por 10.000 épocas com taxa 2.0, mostre as previsões arredondadas e desenhe a curva de perda.

In [ ]:
from gabarito_rover.neural import RedeNeural
rede = RedeNeural(2, 4, 1, semente=1).treinar(X, y, epocas=10000, taxa=2.0)
print(rede.prever(X).round(3).ravel(), "| esperado:", y.ravel())
plt.plot(rede.perdas); plt.yscale("log"); plt.xlabel("época"); plt.ylabel("perda"); plt.show()

### 11.5 **Imagens de verdade.** Carregue `load_digits` do scikit-learn (imagens 8×8 de dígitos, já vêm na biblioteca). Mostre 8 imagens com o rótulo no título (`plt.subplots(1, 8)` e `imshow`).

In [ ]:
from sklearn.datasets import load_digits
dig = load_digits()
print(dig.images.shape)
fig, eixos = plt.subplots(1, 8, figsize=(10, 2))
for ax, img, rot in zip(eixos, dig.images, dig.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(rot); ax.axis("off")
plt.show()

### 11.6 **Treine um classificador.** Separe treino e teste (`train_test_split`, 25% de teste, `random_state=0`), padronize (`StandardScaler`) e treine um `MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=0)`. Mostre a acurácia no teste.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
Xtr, Xte, ytr, yte = train_test_split(dig.data, dig.target, test_size=0.25, random_state=0)
esc = StandardScaler().fit(Xtr)
mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=0).fit(esc.transform(Xtr), ytr)
print("acurácia no teste:", round(mlp.score(esc.transform(Xte), yte), 3))

### 11.7 **Onde o modelo erra.** Mostre a matriz de confusão (`ConfusionMatrixDisplay.from_predictions`) e as 6 primeiras imagens que ele errou, com o título `real→previsto`.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
prev = mlp.predict(esc.transform(Xte))
ConfusionMatrixDisplay.from_predictions(yte, prev); plt.show()
erros = np.where(prev != yte)[0][:6]
fig, eixos = plt.subplots(1, max(1, len(erros)), figsize=(8, 2))
for ax, i in zip(np.atleast_1d(eixos), erros):
    ax.imshow(Xte[i].reshape(8, 8), cmap="gray_r"); ax.set_title(f"{yte[i]}→{prev[i]}"); ax.axis("off")
plt.show()

### 11.8 **Sua rede × scikit-learn.** Use a sua `RedeNeural` para diferenciar só os dígitos 0 e 1: filtre os dados, normalize os pixels (÷16), treine `RedeNeural(64, 16, 1)` por 2.000 épocas e calcule a acurácia no teste (previsão > 0,5 = dígito 1).

In [ ]:
mask_tr, mask_te = np.isin(ytr, [0, 1]), np.isin(yte, [0, 1])
Xa, ya = Xtr[mask_tr] / 16, (ytr[mask_tr] == 1).reshape(-1, 1)
Xb, yb = Xte[mask_te] / 16, (yte[mask_te] == 1)
minha = RedeNeural(64, 16, 1, semente=0).treinar(Xa, ya, epocas=2000, taxa=1.0)
print("acurácia da sua rede (0 × 1):", round(float(((minha.prever(Xb).ravel() > 0.5) == yb).mean()), 3))

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
